In [52]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator


In [53]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(description="feedback of the essay")
    score : int = Field(description="score out of 10 given to the essay ", ge=0, le=10) 

In [54]:
model  = ChatOllama(model = "gemma3:latest")

In [55]:
structured_model = model.with_structured_output(EvaluationSchema)

In [56]:
class EssayState(TypedDict):
    essay : str
    language_feedback : EvaluationSchema
    analysis_feedback : EvaluationSchema
    clarity_feedback : EvaluationSchema
    summary : str
    indivisual_score: dict
    mean_score : float


In [57]:
def language_evaluation(state:EssayState)->dict:
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    result = structured_model.invoke(prompt)
    return {
        "language_feedback": {
            "score": result.score,
            "feedback": result.feedback
        }
    }


In [58]:
def analysis_evaluation(state:EssayState)->dict:
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    result = structured_model.invoke(prompt)
    return {
        "analysis_feedback":{
            "score" : result.score,
            "feedback" : result.feedback
        }
    }

In [59]:
def clarity_evaluation(state:EssayState)->dict:
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    result = structured_model.invoke(prompt)
    return {
        "clarity_feedback":{
            "score" : result.score,
            "feedback":result.feedback
        }
    }

In [60]:
def final_evaluation(state: EssayState) -> dict:
    lang = state.get("language_feedback", {"score": 0, "feedback": "No feedback"})
    analysis = state.get("analysis_feedback", {"score": 0, "feedback": "No feedback"})
    clarity = state.get("clarity_feedback", {"score": 0, "feedback": "No feedback"})

    total_score = lang["score"] + analysis["score"] + clarity["score"]
    mean_score = round(total_score / 3, 2)

    prompt = f"i am providing the feedbacks of language, clarity and depth of analysis evaluation of the essay, write the final summarised feedback. Language feedback : {lang["feedback"]} \n depth of analysis feedback : {analysis["feedback"]}\n clarity feedback: {clarity["feedback"]}"
    
    indivisual_score = {"language" : lang["score"], "analysis" : analysis["score"], "clarity":clarity["score"]}

    result = model.invoke(prompt)
    return {
        "mean_score": mean_score,
        "summary": result.content,
        "indivisual_score" : indivisual_score

    }
    

In [61]:
graph = StateGraph(EssayState)
graph.add_node("language_evaluation", language_evaluation)
graph.add_node("analysis_evaluation", analysis_evaluation)
graph.add_node("clarity_evaluation", clarity_evaluation)
graph.add_node("final_evaluation", final_evaluation)

graph.add_edge(START, "language_evaluation")
graph.add_edge(START, "analysis_evaluation")
graph.add_edge(START, "clarity_evaluation")

graph.add_edge("clarity_evaluation","final_evaluation" )
graph.add_edge("analysis_evaluation","final_evaluation" )
graph.add_edge("language_evaluation","final_evaluation" )

graph.add_edge("final_evaluation", END)






In [62]:
workflow = graph.compile()

In [63]:
essay = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""


In [64]:
input_state = {"essay" : essay}
final_state = workflow.invoke(input_state)

print(final_state)

{'essay': 'India and AI Time\n\nNow world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.\n\nIndia have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.\n\nIn farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.\n\nBut problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.\n\nOne more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.\n\n

In [68]:
print(final_state["language_feedback"])

{'score': 5, 'feedback': "This essay demonstrates a basic understanding of the potential and challenges of AI in India, but it needs significant refinement to reach a higher quality. Here’s a breakdown of the feedback:\n\n**Strengths:**\n* **Clear Central Theme:** The essay clearly identifies the topic of AI in India and its potential impact.\n* **Identifies Key Points:** It touches upon relevant aspects like the availability of skilled workers, existing AI implementation by companies, government initiatives, and potential problems.\n* **Simple and Understandable:** The language is accessible and easy to understand for a general audience.\n\n**Weaknesses:**\n* **Lack of Depth and Detail:** The essay is largely descriptive and lacks analysis. It states facts without exploring them deeply. For example, the discussion of ‘smart students’ could be expanded with statistics or specific examples. \n* **Repetitive Language:** Phrases like “AI help…” are repeated excessively, making the writing